# The Only Tutorial you will need for Explanatory Data Analysis

Welcome to the detective phase of Machine Learning. While building predictive models is the glamorous part of AI, **Explanatory Data Analysis** (more commonly known in the industry as *Exploratory Data Analysis* or EDA) is where the actual data science happens. 

If you feed raw, uninspected data directly into a machine learning algorithm, you are flying blind. EDA is the analytical process of summarizing the main characteristics of a dataset, often using statistical graphics and data visualization methods. 

**The goals of EDA are to:**
1. Uncover underlying structures and patterns.
2. Identify anomalies, outliers, and missing data.
3. Test underlying assumptions before applying complex algorithms.
4. Discover which features (variables) actually correlate with your target outcome.

In this tutorial, we will build a complete, professional EDA pipeline from scratch using Python's core data science stack.

In [ ]:
# Cell 2: Environment Setup
# We import the essential libraries for data manipulation and visualization.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set seaborn aesthetic parameters for high-quality analytical plots
sns.set_theme(style="whitegrid", palette="muted")

# Suppress warnings for cleaner output in our notebook environment
import warnings
warnings.filterwarnings('ignore')

## Step 1: Acquiring the Data

To demonstrate EDA analytically, we need a dataset with realistic quirks: correlations, varying distributions, and some noise. 

Instead of relying on an external file that might break, we will synthesize a realistic "Tech Professional Salary" dataset. This dataset will contain numerical features (Age, Years of Experience), categorical features (Education Level), and our target variable (Salary).

In [ ]:
# Cell 4: Generating the Analytical Dataset
np.random.seed(42) # Ensure reproducibility

# Generating 500 samples
n_samples = 500

# 1. Age (Normal-ish distribution, centered around 35)
age = np.random.normal(loc=35, scale=8, size=n_samples).astype(int)

# 2. Years of Experience (Correlated with Age, but with some randomness)
# We ensure experience is logically bounded (can't have more experience than Age - 18)
experience = np.clip(age - 22 + np.random.normal(0, 3, n_samples), a_min=0, a_max=None).astype(int)

# 3. Education Level (Categorical)
education = np.random.choice(['Bachelors', 'Masters', 'PhD'], p=[0.6, 0.3, 0.1], size=n_samples)

# 4. Salary (Our target variable. Highly dependent on experience and education)
# Base salary is 60k. Each year of exp adds ~5k. Masters adds ~15k, PhD adds ~30k.
base_salary = 60000
edu_bonus = np.where(education == 'Masters', 15000, np.where(education == 'PhD', 30000, 0))
salary = base_salary + (experience * 5000) + edu_bonus + np.random.normal(0, 10000, n_samples)

# Let's inject a few massive outliers into Salary to represent executives/founders
salary[np.random.randint(0, n_samples, 5)] += np.random.uniform(150000, 300000, 5)

# Assemble into a Pandas DataFrame
df = pd.DataFrame({
    'Age': age,
    'Experience': experience,
    'Education': education,
    'Salary': salary
})

print("Dataset successfully generated!")

## Step 2: High-Level Inspection

Before diving into complex statistics, we must simply look at the shape and surface-level details of our data. 
* How many rows and columns do we have? 
* What are the data types? (Are numbers accidentally being read as text?)
* Do we have null values?

In [ ]:
# Cell 6: Structural Analysis

print("--- DataFrame Shape ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

print("--- DataFrame Info (Data Types & Non-Null Counts) ---")
# df.info() is arguably the most important first command in EDA
df.info()

print("\n--- First 5 Rows ---")
# Displaying the head gives us a qualitative feel for the data format
display(df.head())

## Step 3: Descriptive Statistics

Now we move from structure to mathematics. Descriptive statistics provide a numerical summary of our continuous variables. 

**Analytical points to look for here:**
* **Mean vs. Median (50%):** If the mean is significantly higher than the median, your data is right-skewed (likely due to outliers).
* **Min/Max:** Do these values make logical sense? (e.g., An 'Age' of -5 or 999 is a data entry error that needs cleansing).
* **Standard Deviation (std):** How spread out is the data around the mean?

In [ ]:
# Cell 8: Statistical Summary

# df.describe() automatically calculates statistics for all numerical columns
stats_summary = df.describe().T # Transposed for easier reading

# Let's add a custom analytical metric: the Interquartile Range (IQR)
# The IQR is crucial for understanding the spread of the middle 50% of our data
stats_summary['IQR'] = stats_summary['75%'] - stats_summary['25%']

print("--- Descriptive Statistics ---")
display(stats_summary)

# Analytical Observation: Look at 'Salary'. 
# The maximum value is likely vastly higher than the 75th percentile. 
# This mathematically confirms the presence of the executive outliers we injected.

## Step 4: Univariate Analysis (Visualizing Distributions)

Numbers can be deceiving. Visualizing single variables (Univariate Analysis) helps us instantly recognize distributions (Normal, Bimodal, Skewed) and spot outliers.

The two best tools for this are:
1. **Histograms / KDE Plots:** To see the shape of the data.
2. **Boxplots:** To rigorously identify mathematical outliers using quartiles.

In [ ]:
# Cell 10: Univariate Visualization

# We create a figure with 1 row and 2 columns
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: The Distribution of Salary
# KDE (Kernel Density Estimate) smooths the histogram to show the probability density curve
sns.histplot(df['Salary'], bins=30, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Salary Distribution (Note the right tail)', fontsize=14)
axes[0].set_xlabel('Salary ($)')
axes[0].set_ylabel('Frequency')

# Plot 2: Boxplot of Salary
# Boxplots visually represent the IQR. Any dots outside the "whiskers" are mathematical outliers.
sns.boxplot(x=df['Salary'], ax=axes[1], color='lightgreen')
axes[1].set_title('Salary Boxplot (Outliers explicitly visible)', fontsize=14)
axes[1].set_xlabel('Salary ($)')

plt.tight_layout()
plt.show()

## Step 5: Bivariate and Multivariate Analysis (Finding Relationships)

This is the core of predictive modeling prep. We need to know how our features relate to our target variable (`Salary`) and to each other.

If two features correlate perfectly with each other (e.g., "Age" and "Years Since Birth"), they provide redundant information to a model. This is called **Multicollinearity**, and we use a Correlation Matrix to hunt it down.

In [ ]:
# Cell 12: Correlation and Relationship Analysis

# 1. The Correlation Heatmap
# We only calculate correlation for numerical columns.
# Values range from -1 (perfect inverse relationship) to 1 (perfect direct relationship). 0 means no relationship.
plt.figure(figsize=(8, 6))
correlation_matrix = df[['Age', 'Experience', 'Salary']].corr()

# Annot=True prints the numbers inside the boxes. cmap dictates the color scale.
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=16)
plt.show()

# Analytical Observation from Heatmap:
# Experience and Age are highly correlated (~0.9+). 
# In strict linear regression, keeping both might cause multicollinearity. We might consider dropping one.

# 2. Multivariate Visual: Scatterplot with Categorical Hues
# How does Education impact the relationship between Experience and Salary?
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Experience', y='Salary', hue='Education', palette='Set2', alpha=0.8, s=60)
plt.title('Salary vs. Experience across Education Levels', fontsize=14)
plt.xlabel('Years of Experience')
plt.ylabel('Salary ($)')
plt.show()

## Conclusion of the EDA Phase

Through this brief but robust Explanatory Data Analysis, we have successfully profiled our dataset:

1. **Integrity:** We verified our data has no missing values and correct data types.
2. **Distribution:** We found that Salary is positively skewed due to executive outliers. (We would need to address this in the Cleansing phase using clipping or log-transformations before feeding it to a linear model).
3. **Relationships:** We discovered a strong linear relationship between Experience and Salary, with stratified baselines based on Education level.
4. **Redundancy:** We identified a very high correlation between Age and Experience, meaning we only truly need one of them to predict Salary effectively.

By writing these analytical observations down, you transition from someone who just writes code to a true Data Scientist. Your data is now understood, documented, and ready for Feature Engineering!